# 🚀 NIFTY 50 Complete Model Training

**One-click training pipeline for NIFTY 50 Index Prediction**

This notebook combines all training steps:
1. ✅ Data Collection (OHLCV + Sentiment)
2. ✅ Feature Engineering (Technical Indicators)
3. ✅ Model Training (Multimodal TCN)
4. ✅ Export & Download

---

## ⚠️ Instructions

1. **Enable GPU**: Go to `Runtime` → `Change runtime type` → Select `T4 GPU`
2. **Run All**: Go to `Runtime` → `Run all` (or press `Ctrl+F9`)
3. **Wait**: Training takes ~30-60 minutes
4. **Download**: Model will automatically download at the end

---

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install "numpy<2.0" "pandas==2.2.2" yfinance feedparser pandas-ta scikit-learn torch torchvision tqdm matplotlib -q
print("✅ All dependencies installed!")

In [ ]:
# Imports
import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
from sklearn.preprocessing import StandardScaler
from datetime import datetime, timedelta
import feedparser
import json
import os
import pickle
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")
if device.type == 'cuda':
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: No GPU detected! Training will be slower.")

# Create directories
os.makedirs('data', exist_ok=True)
os.makedirs('export', exist_ok=True)

---
## 📊 Step 2: Data Collection

Fetching 3 years of NIFTY 50 historical data...

In [ ]:
# NIFTY 50 Index symbol
SYMBOL = "^NSEI"

# Fetch 3 years of data
end_date = datetime.now()
start_date = end_date - timedelta(days=365 * 3)

print(f"📅 Fetching data from {start_date.date()} to {end_date.date()}")

df_nifty = yf.download(
    SYMBOL,
    start=start_date.strftime('%Y-%m-%d'),
    end=end_date.strftime('%Y-%m-%d'),
    progress=True
)

print(f"\n✅ Downloaded {len(df_nifty)} trading days")
df_nifty.to_csv('data/nifty50_ohlcv.csv')
df_nifty.tail()

In [ ]:
# Fetch top constituents data
print("📈 Fetching top NIFTY 50 constituents...")

CONSTITUENTS = [
    "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "INFY.NS", "ICICIBANK.NS",
    "HINDUNILVR.NS", "BHARTIARTL.NS", "SBIN.NS", "BAJFINANCE.NS", "ITC.NS",
]

constituents_data = {}
for symbol in CONSTITUENTS:
    try:
        df = yf.download(symbol, start=start_date.strftime('%Y-%m-%d'), 
                         end=end_date.strftime('%Y-%m-%d'), progress=False)
        constituents_data[symbol] = df
        filename = f"data/{symbol.replace('.NS', '')}_ohlcv.csv"
        df.to_csv(filename)
        print(f"  ✓ {symbol}: {len(df)} days")
    except Exception as e:
        print(f"  ✗ {symbol}: {e}")

print(f"\n✅ Fetched {len(constituents_data)} constituents")

In [ ]:
# Generate synthetic sentiment for training
print("🎭 Generating sentiment data...")

def generate_synthetic_sentiment(df_prices):
    """Generate synthetic sentiment scores based on price movements."""
    sentiment_data = []
    
    for i in range(1, len(df_prices)):
        current_date = df_prices.index[i]
        prev_close = df_prices['Close'].iloc[i-1]
        curr_close = df_prices['Close'].iloc[i]
        
        daily_return = (curr_close - prev_close) / prev_close
        base_sentiment = np.tanh(daily_return * 50)
        noise = np.random.normal(0, 0.1)
        sentiment = np.clip(base_sentiment + noise, -1, 1)
        
        sentiment_data.append({
            'date': current_date.strftime('%Y-%m-%d'),
            'news_sentiment': float(sentiment * 0.8 + np.random.normal(0, 0.1)),
            'reddit_sentiment': float(sentiment * 0.6 + np.random.normal(0, 0.15)),
            'combined_sentiment': float(sentiment),
        })
    
    return pd.DataFrame(sentiment_data)

df_sentiment = generate_synthetic_sentiment(df_nifty)
df_sentiment.to_csv('data/sentiment_scores.csv', index=False)
print(f"✅ Generated {len(df_sentiment)} sentiment records")

---
## ⚙️ Step 3: Feature Engineering

Computing technical indicators...

In [ ]:
# Load NIFTY 50 data
df = pd.read_csv('data/nifty50_ohlcv.csv', index_col=0, parse_dates=True)
print(f"📊 Loaded {len(df)} trading days")

def compute_technical_indicators(df):
    """Compute all technical indicators using pandas-ta."""
    df = df.copy()
    
    # Momentum Indicators
    df['rsi_14'] = ta.rsi(df['Close'], length=14)
    
    macd = ta.macd(df['Close'])
    if macd is not None:
        df['macd'] = macd['MACD_12_26_9']
        df['macd_signal'] = macd['MACDs_12_26_9']
        df['macd_hist'] = macd['MACDh_12_26_9']
    
    stoch = ta.stoch(df['High'], df['Low'], df['Close'])
    if stoch is not None:
        df['stoch_k'] = stoch['STOCHk_14_3_3']
        df['stoch_d'] = stoch['STOCHd_14_3_3']
    
    # Trend Indicators
    df['ema_5'] = ta.ema(df['Close'], length=5)
    df['ema_20'] = ta.ema(df['Close'], length=20)
    df['ema_50'] = ta.ema(df['Close'], length=50)
    df['sma_20'] = ta.sma(df['Close'], length=20)
    
    adx = ta.adx(df['High'], df['Low'], df['Close'])
    if adx is not None:
        df['adx'] = adx['ADX_14']
    
    # Volatility Indicators
    df['atr_14'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)
    
    bbands = ta.bbands(df['Close'])
    if bbands is not None:
        df['bb_upper'] = bbands['BBU_5_2.0']
        df['bb_middle'] = bbands['BBM_5_2.0']
        df['bb_lower'] = bbands['BBL_5_2.0']
    
    # Volume Indicators
    df['obv'] = ta.obv(df['Close'], df['Volume'])
    df['volume_sma'] = ta.sma(df['Volume'], length=20)
    
    return df

df_with_ta = compute_technical_indicators(df)
print(f"✅ Added {len(df_with_ta.columns) - len(df.columns)} technical indicators")

In [ ]:
# Create target variables
def create_targets(df, prediction_horizon=1):
    df = df.copy()
    df['next_close'] = df['Close'].shift(-prediction_horizon)
    df['return_pct'] = (df['next_close'] - df['Close']) / df['Close'] * 100
    df['direction'] = 0
    df.loc[df['return_pct'] > 0.3, 'direction'] = 1
    df.loc[df['return_pct'] < -0.3, 'direction'] = -1
    return df

df_with_targets = create_targets(df_with_ta)
print(f"📈 Target distribution:\n{df_with_targets['direction'].value_counts()}")

In [ ]:
# Merge with sentiment
df_sentiment = pd.read_csv('data/sentiment_scores.csv', parse_dates=['date'])
df_sentiment.set_index('date', inplace=True)
df_merged = df_with_targets.join(df_sentiment, how='left')
df_merged[['news_sentiment', 'reddit_sentiment', 'combined_sentiment']] = \
    df_merged[['news_sentiment', 'reddit_sentiment', 'combined_sentiment']].fillna(0)

print(f"✅ Merged dataset: {len(df_merged)} rows")

In [ ]:
# Create sequences for TCN
def create_sequences(df, seq_length=60):
    df_clean = df.dropna()
    
    price_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    tech_cols = [
        'rsi_14', 'macd', 'macd_signal', 'macd_hist', 
        'stoch_k', 'stoch_d', 'ema_5', 'ema_20', 'ema_50',
        'sma_20', 'adx', 'atr_14', 'bb_upper', 'bb_middle', 'bb_lower'
    ]
    sent_cols = ['news_sentiment', 'reddit_sentiment', 'combined_sentiment']
    
    price_scaler = StandardScaler()
    price_data = price_scaler.fit_transform(df_clean[price_cols])
    
    tech_scaler = StandardScaler()
    tech_data = tech_scaler.fit_transform(df_clean[tech_cols].fillna(0))
    
    sent_data = df_clean[sent_cols].values
    targets = df_clean['return_pct'].values
    
    price_sequences = []
    technical_features = []
    sentiment_features = []
    target_values = []
    
    for i in range(seq_length, len(df_clean) - 1):
        price_sequences.append(price_data[i-seq_length:i])
        technical_features.append(tech_data[i])
        sentiment_features.append(sent_data[i])
        target_values.append(targets[i])
    
    return (
        np.array(price_sequences),
        np.array(technical_features),
        np.array(sentiment_features),
        np.array(target_values),
        price_scaler,
        tech_scaler
    )

SEQ_LENGTH = 60
price_seq, tech_feat, sent_feat, targets, price_scaler, tech_scaler = create_sequences(
    df_merged, seq_length=SEQ_LENGTH
)

print(f"📊 Dataset shapes:")
print(f"  Price sequences: {price_seq.shape}")
print(f"  Technical features: {tech_feat.shape}")
print(f"  Sentiment features: {sent_feat.shape}")
print(f"  Targets: {targets.shape}")

In [ ]:
# Train/Validation/Test Split (time-based, no shuffling!)
train_size = int(len(targets) * 0.7)
val_size = int(len(targets) * 0.15)

X_price_train = price_seq[:train_size]
X_price_val = price_seq[train_size:train_size+val_size]
X_price_test = price_seq[train_size+val_size:]

X_tech_train = tech_feat[:train_size]
X_tech_val = tech_feat[train_size:train_size+val_size]
X_tech_test = tech_feat[train_size+val_size:]

X_sent_train = sent_feat[:train_size]
X_sent_val = sent_feat[train_size:train_size+val_size]
X_sent_test = sent_feat[train_size+val_size:]

y_train = targets[:train_size]
y_val = targets[train_size:train_size+val_size]
y_test = targets[train_size+val_size:]

print(f"✅ Split sizes:")
print(f"  Train: {len(y_train)}")
print(f"  Validation: {len(y_val)}")
print(f"  Test: {len(y_test)}")

# Save scalers
with open('data/price_scaler.pkl', 'wb') as f:
    pickle.dump(price_scaler, f)
with open('data/tech_scaler.pkl', 'wb') as f:
    pickle.dump(tech_scaler, f)

---
## 🧠 Step 4: Model Architecture

Defining the multimodal TCN model...

In [ ]:
class TemporalBlock(nn.Module):
    """TCN block with dilated causal convolution."""
    
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.2):
        super().__init__()
        
        padding = (kernel_size - 1) * dilation
        
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size,
                               padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size,
                               padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        
    def forward(self, x):
        out = self.conv1(x)
        out = out[:, :, :-self.conv1.padding[0]] if self.conv1.padding[0] > 0 else out
        out = self.relu(self.bn1(out))
        out = self.dropout(out)
        
        out = self.conv2(out)
        out = out[:, :, :-self.conv2.padding[0]] if self.conv2.padding[0] > 0 else out
        out = self.relu(self.bn2(out))
        out = self.dropout(out)
        
        res = self.downsample(x) if self.downsample else x
        if res.size(2) > out.size(2):
            res = res[:, :, :out.size(2)]
        elif res.size(2) < out.size(2):
            out = out[:, :, :res.size(2)]
            
        return self.relu(out + res)


class TCNEncoder(nn.Module):
    """TCN for price time-series encoding."""
    
    def __init__(self, input_features=5, hidden_channels=64, embedding_dim=128, dropout=0.2):
        super().__init__()
        
        self.input_projection = nn.Conv1d(input_features, hidden_channels, 1)
        
        dilations = [1, 2, 4, 8, 16, 32]
        layers = [TemporalBlock(hidden_channels, hidden_channels, 3, d, dropout) for d in dilations]
        self.tcn = nn.Sequential(*layers)
        
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.projection = nn.Sequential(
            nn.Linear(hidden_channels, embedding_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        
    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.input_projection(x)
        x = self.tcn(x)
        x = self.global_pool(x).squeeze(-1)
        return self.projection(x)


class TechnicalEncoder(nn.Module):
    """MLP for technical indicators."""
    
    def __init__(self, input_features=15, embedding_dim=128, dropout=0.2):
        super().__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, embedding_dim),
            nn.ReLU(),
        )
        
    def forward(self, x):
        return self.encoder(x)


class SentimentEncoder(nn.Module):
    """Simple encoder for sentiment scores."""
    
    def __init__(self, input_features=3, embedding_dim=128, dropout=0.2):
        super().__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_features, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, embedding_dim),
            nn.ReLU(),
        )
        
    def forward(self, x):
        return self.encoder(x)


class AdaptiveFusionGate(nn.Module):
    """Dynamic modality weighting."""
    
    def __init__(self, embedding_dim=128, num_modalities=3, dropout=0.2):
        super().__init__()
        
        self.context_encoder = nn.Sequential(
            nn.Linear(embedding_dim * num_modalities, embedding_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embedding_dim, num_modalities),
        )
        self.temperature = nn.Parameter(torch.ones(1))
        
    def forward(self, price_emb, sentiment_emb, technical_emb):
        embeddings = torch.stack([price_emb, sentiment_emb, technical_emb], dim=1)
        context = torch.cat([price_emb, sentiment_emb, technical_emb], dim=-1)
        
        logits = self.context_encoder(context)
        weights = torch.softmax(logits / self.temperature, dim=-1)
        
        weighted = embeddings * weights.unsqueeze(-1)
        fused = weighted.view(weights.size(0), -1)
        
        return fused, weights


class PredictionHead(nn.Module):
    """Multi-output prediction head."""
    
    def __init__(self, input_dim=384, hidden_dim=256, dropout=0.3):
        super().__init__()
        
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        
        self.point_head = nn.Linear(hidden_dim // 2, 1)
        self.quantile_head = nn.Linear(hidden_dim // 2, 3)
        
    def forward(self, x):
        shared = self.shared(x)
        point = self.point_head(shared).squeeze(-1)
        quantiles = self.quantile_head(shared)
        
        return {
            'point': point,
            'quantile_5': quantiles[:, 0],
            'quantile_50': quantiles[:, 1],
            'quantile_95': quantiles[:, 2],
        }


class NIFTY50Predictor(nn.Module):
    """Complete multimodal model."""
    
    def __init__(self, embedding_dim=128, dropout=0.2):
        super().__init__()
        
        self.price_encoder = TCNEncoder(5, 64, embedding_dim, dropout)
        self.sentiment_encoder = SentimentEncoder(3, embedding_dim, dropout)
        self.technical_encoder = TechnicalEncoder(15, embedding_dim, dropout)
        self.fusion = AdaptiveFusionGate(embedding_dim, 3, dropout)
        self.prediction_head = PredictionHead(embedding_dim * 3, 256, dropout)
        
    def forward(self, price, sentiment, technical):
        price_emb = self.price_encoder(price)
        sent_emb = self.sentiment_encoder(sentiment)
        tech_emb = self.technical_encoder(technical)
        
        fused, weights = self.fusion(price_emb, sent_emb, tech_emb)
        predictions = self.prediction_head(fused)
        predictions['modality_weights'] = weights
        
        return predictions

print("✅ Model architecture defined!")

---
## 🏋️ Step 5: Training

Training the model for 50 epochs...

In [ ]:
# Dataset and DataLoader
class NIFTY50Dataset(Dataset):
    def __init__(self, price, technical, sentiment, targets):
        self.price = torch.tensor(price, dtype=torch.float32)
        self.technical = torch.tensor(technical, dtype=torch.float32)
        self.sentiment = torch.tensor(sentiment, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)
        
    def __len__(self):
        return len(self.targets)
    
    def __getitem__(self, idx):
        return {
            'price': self.price[idx],
            'technical': self.technical[idx],
            'sentiment': self.sentiment[idx],
            'target': self.targets[idx],
        }

train_dataset = NIFTY50Dataset(X_price_train, X_tech_train, X_sent_train, y_train)
val_dataset = NIFTY50Dataset(X_price_val, X_tech_val, X_sent_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"✅ DataLoaders created")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")

In [ ]:
# Training functions
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    
    for batch in tqdm(loader, desc="Training", leave=False):
        optimizer.zero_grad()
        
        price = batch['price'].to(device)
        technical = batch['technical'].to(device)
        sentiment = batch['sentiment'].to(device)
        target = batch['target'].to(device)
        
        output = model(price, sentiment, technical)
        
        loss = criterion(output['point'], target)
        loss += 0.5 * criterion(output['quantile_50'], target)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    predictions = []
    actuals = []
    
    with torch.no_grad():
        for batch in loader:
            price = batch['price'].to(device)
            technical = batch['technical'].to(device)
            sentiment = batch['sentiment'].to(device)
            target = batch['target'].to(device)
            
            output = model(price, sentiment, technical)
            loss = criterion(output['point'], target)
            
            total_loss += loss.item()
            predictions.extend(output['point'].cpu().numpy())
            actuals.extend(target.cpu().numpy())
    
    pred_dir = np.sign(predictions)
    actual_dir = np.sign(actuals)
    direction_acc = (pred_dir == actual_dir).mean()
    
    return total_loss / len(loader), direction_acc

In [ ]:
# Initialize model and start training
model = NIFTY50Predictor(embedding_dim=128, dropout=0.2).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
criterion = nn.MSELoss()

print(f"🧠 Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"🚀 Starting training...\n")

EPOCHS = 50
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    status = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'nifty50_model.pt')
        status = " ✅ Saved!"
    
    print(f"Epoch {epoch+1:2d}/{EPOCHS}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, Dir Acc={val_acc:.2%}{status}")

print(f"\n🎉 Training complete! Best Val Loss: {best_val_loss:.4f}")

In [ ]:
# Visualize training
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train', linewidth=2)
ax1.plot(history['val_loss'], label='Validation', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['val_acc'], color='green', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Direction Prediction Accuracy')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()
print("✅ Training history saved!")

In [ ]:
# Evaluate on test set
test_dataset = NIFTY50Dataset(X_price_test, X_tech_test, X_sent_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Load best model
model.load_state_dict(torch.load('nifty50_model.pt'))
test_loss, test_acc = validate(model, test_loader, criterion, device)

print(f"\n{'='*50}")
print(f"📊 TEST RESULTS")
print(f"{'='*50}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Direction Accuracy: {test_acc:.2%}")
print(f"{'='*50}")

---
## 📦 Step 6: Export & Download

Preparing model for deployment...

In [ ]:
# Export model
model_cpu = model.cpu()
model_cpu.eval()

# Test inference
batch_size = 1
price_dummy = torch.randn(batch_size, 60, 5)
sentiment_dummy = torch.randn(batch_size, 3)
technical_dummy = torch.randn(batch_size, 15)

with torch.no_grad():
    output = model_cpu(price_dummy, sentiment_dummy, technical_dummy)

print("🧪 Test inference output:")
print(f"  Point prediction: {output['point'].item():.4f}%")
print(f"  Quantile 5%: {output['quantile_5'].item():.4f}%")
print(f"  Quantile 95%: {output['quantile_95'].item():.4f}%")
print(f"  Modality weights: {output['modality_weights'].numpy().round(3)}")

In [ ]:
# Save model files
torch.save(model_cpu.state_dict(), 'export/nifty50_model.pt')
print("✅ Saved: export/nifty50_model.pt")

# Copy scalers
shutil.copy('data/price_scaler.pkl', 'export/price_scaler.pkl')
shutil.copy('data/tech_scaler.pkl', 'export/tech_scaler.pkl')
print("✅ Saved: scalers")

# Copy training history
shutil.copy('training_history.png', 'export/training_history.png')
print("✅ Saved: training_history.png")

# Create model info
model_info = {
    "name": "NIFTY50Predictor",
    "version": "1.0.0",
    "trained_on": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "test_accuracy": f"{test_acc:.2%}",
    "architecture": {
        "price_encoder": "TCN (6 blocks, dilations 1-32)",
        "sentiment_encoder": "MLP (3 → 64 → 128)",
        "technical_encoder": "MLP (15 → 64 → 128)",
        "fusion": "Adaptive Gate (3 modalities)",
        "prediction_head": "MLP with quantile outputs",
    },
    "input_shapes": {
        "price": [60, 5],
        "sentiment": [3],
        "technical": [15],
    },
    "parameters": sum(p.numel() for p in model_cpu.parameters()),
}

with open('export/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)
print("✅ Saved: model_info.json")

In [ ]:
# Create zip and download
shutil.make_archive('nifty50_model_export', 'zip', 'export')

print("\n" + "="*60)
print("🎉 TRAINING COMPLETE!")
print("="*60)
print(f"")
print(f"📊 Final Results:")
print(f"   • Best Validation Loss: {best_val_loss:.4f}")
print(f"   • Test Direction Accuracy: {test_acc:.2%}")
print(f"   • Model Parameters: {sum(p.numel() for p in model_cpu.parameters()):,}")
print(f"")
print(f"📦 Export contents:")
print(f"   • nifty50_model.pt - Model weights")
print(f"   • price_scaler.pkl - Price data scaler")
print(f"   • tech_scaler.pkl - Technical indicators scaler")
print(f"   • model_info.json - Model metadata")
print(f"   • training_history.png - Training charts")
print(f"")
print("="*60)
print("⬇️ DOWNLOADING MODEL...")
print("="*60)

# Download
from google.colab import files
files.download('nifty50_model_export.zip')

---
## ✅ Next Steps

After downloading `nifty50_model_export.zip`:

1. **Extract** the zip file
2. **Copy** `nifty50_model.pt` to your project's `models/` folder
3. **Copy** the scaler files (`price_scaler.pkl`, `tech_scaler.pkl`) to `models/`
4. **Run** the application with `run_all.bat`

---

🎉 **Congratulations! Your NIFTY 50 prediction model is ready!**